# Reviderad modellträning för Bostadskollen

Filen kan öppnas i VS Code och köras cell för cell som en notebook.
Testdatan används inte innan en slutmodell har valts.



## Importera paket



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder




## Läs in datasetet

Vi behåller originalet oförändrat och arbetar med en kopia.



In [2]:
DATA_PATH = Path("../data/SwedenHousingPrices.csv")

df_original = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df = df_original.copy()

print(f"Antal rader i rådata: {len(df)}")
df.head()




Antal rader i rådata: 11549


,ad_id,date_published,typology,asking_price_sek,land_area_sqm,living_area_sqm,sqm_price_sek,number_rooms,address,location,coordenates
0,21440984,2024-12-28,APARTMENT,3695000.0,0.0,82.0,45061.0,4.0,Einar Hansens esplanad 14,"Västra Hamnen, Malmö kommun","55.6118431,12.9822884"
1,21462070,2024-12-28,APARTMENT,3095000.0,0.0,109.0,28394.0,4.0,Skarpskyttevägen 30A,"Norra Fäladen, Lunds kommun","55.7230072,13.1973028"
2,21455084,2024-12-28,APARTMENT,2295000.0,0.0,54.0,42500.0,2.0,Boplatsvägen 9,"Brotorp/järvastaden, Sundbybergs kommun","59.381943,17.973993"
3,21411619,2024-12-28,APARTMENT,3675000.0,0.0,119.0,30882.0,4.0,Drottninggatan 34C,"Alingsås, Alingsås kommun","57.93111,12.5369"
4,21461458,2024-12-28,APARTMENT,295000.0,0.0,99.0,2980.0,4.0,Valhallagatan 19A,"Skara, Skara kommun","58.38283,13.4190693"


## Behåll bostadstyperna som appen stödjer



In [3]:
allowed_typologies = ["APARTMENT", "HOUSE", "ROW_HOUSE"]
df = df[df["typology"].isin(allowed_typologies)].copy()

df["typology"].value_counts()




typology
APARTMENT    7269
HOUSE        2647
ROW_HOUSE     590
Name: count, dtype: int64

## Skapa koordinater och kommun

Kommunen hämtas från den sista delen av `location`.



In [4]:
df[["latitude", "longitude"]] = (
    df["coordenates"]
    .str.split(",", expand=True)
    .astype(float)
)

df["municipality"] = (
    df["location"]
    .str.split(", ")
    .str[-1]
    .str.strip()
)

df[["location", "municipality", "latitude", "longitude"]].head()




,location,municipality,latitude,longitude
0,"Västra Hamnen, Malmö kommun",Malmö kommun,55.611843,12.982288
1,"Norra Fäladen, Lunds kommun",Lunds kommun,55.723007,13.197303
2,"Brotorp/järvastaden, Sundbybergs kommun",Sundbybergs kommun,59.381943,17.973993
3,"Alingsås, Alingsås kommun",Alingsås kommun,57.931110,12.536900
4,"Skara, Skara kommun",Skara kommun,58.382830,13.419069


## Grundläggande datatvätt

Vi tar bort ogiltiga värden och avgränsar modellen till utgångspriser mellan
100 000 och 10 miljoner kronor.



In [5]:
rows_before = len(df)

df = df[
    df["asking_price_sek"].between(100_000, 10_000_000)
    & (df["living_area_sqm"] > 0)
    & (df["number_rooms"] > 0)
    & df["latitude"].between(55.0, 69.1)
    & df["longitude"].between(10.5, 24.2)
].copy()

print(f"Borttagna rader: {rows_before - len(df)}")
print(f"Kvarvarande rader: {len(df)}")




Borttagna rader: 301
Kvarvarande rader: 10205


## Bostadstypsspecifika gränser

Reglerna avgränsar modellen från extrema specialobjekt.



In [6]:
apartment_mask = (
    (df["typology"] == "APARTMENT")
    & df["living_area_sqm"].between(10, 300)
    & df["number_rooms"].between(1, 10)
)

house_mask = (
    (df["typology"] == "HOUSE")
    & df["living_area_sqm"].between(25, 400)
    & df["number_rooms"].between(1, 15)
    & (df["land_area_sqm"] <= 4_000)
)

row_house_mask = (
    (df["typology"] == "ROW_HOUSE")
    & df["living_area_sqm"].between(40, 300)
    & df["number_rooms"].between(1, 10)
)

df = df[apartment_mask | house_mask | row_house_mask].copy()

df["typology"].value_counts()




typology
APARTMENT    7199
HOUSE        2313
ROW_HOUSE     578
Name: count, dtype: int64

## Hantera tomtarea

Tomtarea används inte för lägenheter. Villavärden under 50 m² är osäkra och
behandlas som saknade i stället för att vi gissar en enhet.



In [7]:
df["has_land_area"] = (df["land_area_sqm"] > 0).astype(int)

df.loc[
    df["typology"] == "APARTMENT",
    "land_area_sqm"
] = np.nan

df.loc[
    (df["typology"] == "HOUSE")
    & (df["land_area_sqm"] < 50),
    "land_area_sqm"
] = np.nan

print(f"Antal rader efter all tvätt: {len(df)}")




Antal rader efter all tvätt: 10090


## Dela data i 60 % train, 20 % validation och 20 % test

Först avsätts 20 % till test. Därefter delas återstående data så att 20 % av
hela datasetet blir validation. `stratify` bevarar fördelningen av bostadstyper.



In [8]:
train_validation_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["typology"]
)

train_df, validation_df = train_test_split(
    train_validation_df,
    test_size=0.25,
    random_state=42,
    stratify=train_validation_df["typology"]
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(validation_df)}")
print(f"Test: {len(test_df)}")




Train: 6054
Validation: 2018
Test: 2018


## Kontrollera fördelningen av bostadstyper

Tabellen visar hur många lägenheter, villor och radhus som finns i varje del.



In [9]:
split_distribution = pd.DataFrame({
    "Train": train_df["typology"].value_counts(),
    "Validation": validation_df["typology"].value_counts(),
    "Test": test_df["typology"].value_counts()
})

split_distribution




,Train,Validation,Test
typology,,,
APARTMENT,4319,1440,1440
HOUSE,1388,462,463
ROW_HOUSE,347,116,115


## Funktion för modellernas resultat



In [10]:
def calculate_metrics(y_true, predictions):
    return {
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "Median error": median_absolute_error(y_true, predictions),
        "R2": r2_score(y_true, predictions)
    }




## Medianbaseline

Baseline gissar alltid medianpriset. ML-modellerna ska slå detta resultat.



In [11]:
baseline = DummyRegressor(strategy="median")

baseline.fit(
    np.zeros((len(train_df), 1)),
    train_df["asking_price_sek"]
)

baseline_predictions = baseline.predict(
    np.zeros((len(validation_df), 1))
)

baseline_metrics = calculate_metrics(
    validation_df["asking_price_sek"],
    baseline_predictions
)

pd.DataFrame([baseline_metrics], index=["Median baseline"])




,MAE,RMSE,Median error,R2
Median baseline,1.299588e+06,1.875014e+06,900000.0,-0.075383


## Välj features

Den globala modellen använder bostadstyp. Specialmodellerna behöver inte den
kolumnen eftersom de tränas på en bostadstyp i taget.



In [12]:
global_numeric_features = [
    "land_area_sqm",
    "living_area_sqm",
    "number_rooms",
    "latitude",
    "longitude",
    "has_land_area"
]

global_categorical_features = ["municipality", "typology"]

apartment_numeric_features = [
    "living_area_sqm",
    "number_rooms",
    "latitude",
    "longitude"
]

property_numeric_features = [
    "land_area_sqm",
    "living_area_sqm",
    "number_rooms",
    "latitude",
    "longitude",
    "has_land_area"
]

specialized_categorical_features = ["municipality"]




## Preprocessing för den globala modellen

Saknade numeriska värden fylls med medianen. Kategorier one-hot-encodas.



In [13]:
global_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median", add_indicator=True),
        global_numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        global_categorical_features
    )
])




## Preprocessing för lägenheter



In [14]:
apartment_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        apartment_numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        specialized_categorical_features
    )
])




## Preprocessing för villor och radhus



In [15]:
property_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median", add_indicator=True),
        property_numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        specialized_categorical_features
    )
])




## Modeller som ska jämföras

Detta är en första screening. Tuning görs efter modelljämförelsen.



In [16]:
models = {
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=300,
        l2_regularization=1.0,
        random_state=42
    )
}




## Funktion som jämför modellerna

Varje modell tränas på train och utvärderas på validation. Test används inte.



In [17]:
def compare_models(
    segment_name,
    train_data,
    validation_data,
    feature_columns,
    preprocessor
):
    X_train = train_data[feature_columns]
    y_train = train_data["asking_price_sek"]

    X_validation = validation_data[feature_columns]
    y_validation = validation_data["asking_price_sek"]

    results = []

    for model_name, model in models.items():
        pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", clone(model))
        ])

        pipeline.fit(X_train, y_train)

        train_predictions = pipeline.predict(X_train)
        validation_predictions = pipeline.predict(X_validation)

        train_rmse = np.sqrt(
            mean_squared_error(y_train, train_predictions)
        )

        metrics = calculate_metrics(
            y_validation,
            validation_predictions
        )

        results.append({
            "Segment": segment_name,
            "Model": model_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "Median error": metrics["Median error"],
            "R2": metrics["R2"],
            "Train RMSE": train_rmse,
            "RMSE gap": metrics["RMSE"] - train_rmse
        })

    return pd.DataFrame(results)




## Jämför modeller på hela datasetet



In [18]:
global_features = global_numeric_features + global_categorical_features

global_results = compare_models(
    "GLOBAL",
    train_df,
    validation_df,
    global_features,
    global_preprocessor
)

global_results.sort_values("RMSE")




,Segment,Model,MAE,RMSE,Median error,R2,Train RMSE,RMSE gap
3,GLOBAL,HistGradientBoosting,630911.710013,9.599110e+05,397228.458158,0.718151,6.809708e+05,2.789402e+05
1,GLOBAL,Random Forest,623569.836033,9.705846e+05,385741.666667,0.711848,3.475094e+05,6.230752e+05
2,GLOBAL,Extra Trees,638711.640486,1.014360e+06,384083.333333,0.685270,1.116867e+04,1.003192e+06
0,GLOBAL,Ridge,860628.563317,1.233159e+06,578387.955337,0.534851,1.164692e+06,6.846631e+04


## Jämför modeller för lägenheter



In [19]:
apartment_train = train_df[train_df["typology"] == "APARTMENT"].copy()
apartment_validation = validation_df[
    validation_df["typology"] == "APARTMENT"
].copy()

apartment_features = (
    apartment_numeric_features
    + specialized_categorical_features
)

apartment_results = compare_models(
    "APARTMENT",
    apartment_train,
    apartment_validation,
    apartment_features,
    apartment_preprocessor
)

apartment_results.sort_values("RMSE")




,Segment,Model,MAE,RMSE,Median error,R2,Train RMSE,RMSE gap
3,APARTMENT,HistGradientBoosting,465088.119944,7.352084e+05,294091.157126,0.787644,5.154010e+05,219807.427223
1,APARTMENT,Random Forest,469945.176179,7.576309e+05,282250.000000,0.774493,2.710797e+05,486551.172130
2,APARTMENT,Extra Trees,486627.494097,8.070775e+05,290325.000000,0.744098,1.298717e+04,794090.311246
0,APARTMENT,Ridge,739290.353703,1.091074e+06,521346.589190,0.532317,1.016649e+06,74424.591928


## Jämför modeller för villor



In [20]:
house_train = train_df[train_df["typology"] == "HOUSE"].copy()
house_validation = validation_df[
    validation_df["typology"] == "HOUSE"
].copy()

property_features = (
    property_numeric_features
    + specialized_categorical_features
)

house_results = compare_models(
    "HOUSE",
    house_train,
    house_validation,
    property_features,
    property_preprocessor
)

house_results.sort_values("RMSE")




,Segment,Model,MAE,RMSE,Median error,R2,Train RMSE,RMSE gap
1,HOUSE,Random Forest,1.117185e+06,1.485617e+06,895341.666667,0.508402,5.225738e+05,9.630436e+05
3,HOUSE,HistGradientBoosting,1.143684e+06,1.515295e+06,932444.059705,0.488565,6.716514e+05,8.436435e+05
0,HOUSE,Ridge,1.178567e+06,1.538500e+06,986776.277298,0.472780,1.279593e+06,2.589071e+05
2,HOUSE,Extra Trees,1.154706e+06,1.568175e+06,895175.000000,0.452246,4.353157e+03,1.563822e+06


## Jämför modeller för radhus



In [21]:
row_house_train = train_df[
    train_df["typology"] == "ROW_HOUSE"
].copy()
row_house_validation = validation_df[
    validation_df["typology"] == "ROW_HOUSE"
].copy()

row_house_results = compare_models(
    "ROW_HOUSE",
    row_house_train,
    row_house_validation,
    property_features,
    property_preprocessor
)

row_house_results.sort_values("RMSE")




,Segment,Model,MAE,RMSE,Median error,R2,Train RMSE,RMSE gap
1,ROW_HOUSE,Random Forest,702943.844691,9.786389e+05,505641.666667,0.642387,367641.538405,6.109973e+05
3,ROW_HOUSE,HistGradientBoosting,775091.917302,1.069311e+06,579663.115343,0.573051,427760.322867,6.415511e+05
0,ROW_HOUSE,Ridge,786433.093011,1.069999e+06,561150.009318,0.572501,848568.221029,2.214310e+05
2,ROW_HOUSE,Extra Trees,795320.152299,1.142890e+06,546175.000000,0.512273,2354.440012,1.140536e+06


## Samlad modelljämförelse



In [22]:
comparison_df = pd.concat([
    global_results,
    apartment_results,
    house_results,
    row_house_results
], ignore_index=True)

comparison_df = comparison_df.sort_values(["Segment", "RMSE"])
comparison_df.round(0)




,Segment,Model,MAE,RMSE,Median error,R2,Train RMSE,RMSE gap
7,APARTMENT,HistGradientBoosting,465088.0,735208.0,294091.0,1.0,515401.0,219807.0
5,APARTMENT,Random Forest,469945.0,757631.0,282250.0,1.0,271080.0,486551.0
6,APARTMENT,Extra Trees,486627.0,807077.0,290325.0,1.0,12987.0,794090.0
4,APARTMENT,Ridge,739290.0,1091074.0,521347.0,1.0,1016649.0,74425.0
3,GLOBAL,HistGradientBoosting,630912.0,959911.0,397228.0,1.0,680971.0,278940.0
1,GLOBAL,Random Forest,623570.0,970585.0,385742.0,1.0,347509.0,623075.0
2,GLOBAL,Extra Trees,638712.0,1014360.0,384083.0,1.0,11169.0,1003192.0
0,GLOBAL,Ridge,860629.0,1233159.0,578388.0,1.0,1164692.0,68466.0
9,HOUSE,Random Forest,1117185.0,1485617.0,895342.0,1.0,522574.0,963044.0
11,HOUSE,HistGradientBoosting,1143684.0,1515295.0,932444.0,0.0,671651.0,843643.0


## Bästa modell per segment

Vinnaren är modellen med lägst RMSE på validation-setet.



In [23]:
best_models = comparison_df.loc[
    comparison_df.groupby("Segment")["RMSE"].idxmin()
]

best_models = best_models[
    ["Segment", "Model", "MAE", "RMSE", "R2", "RMSE gap"]
].sort_values("Segment")

best_models.round(2)

print("\nBästa modell per segment:")
print(best_models.round(2).to_string(index=False))





Bästa modell per segment:
  Segment                Model        MAE       RMSE   R2  RMSE gap
APARTMENT HistGradientBoosting  465088.12  735208.38 0.79 219807.43
   GLOBAL HistGradientBoosting  630911.71  959911.03 0.72 278940.21
    HOUSE        Random Forest 1117185.08 1485617.33 0.51 963043.58
ROW_HOUSE        Random Forest  702943.84  978638.86 0.64 610997.32


# Hyperparametertuning

Screeningen visade att HistGradientBoosting var bäst för globalmodellen och
lägenheter. Random Forest var bäst för villor och radhus. Vi tunar därför bara
dessa kombinationer.

`GridSearchCV` testar parametrarna med tre delningar av träningsdatan. Testdata
används fortfarande inte.




## Parametrar för HistGradientBoosting

Vi testar tre learning rates, två trädstorlekar och två minsta lövstorlekar.
Det ger totalt 12 kombinationer.



In [24]:
hist_parameter_grid = {
    "model__learning_rate": [0.03, 0.05, 0.08],
    "model__max_leaf_nodes": [15, 31],
    "model__min_samples_leaf": [10, 20]
}




## Parametrar för Random Forest

Vi testar två träddjup, tre minsta lövstorlekar och två feature-nivåer.
Antalet träd hålls på 400 för stabila resultat.



In [25]:
random_forest_parameter_grid = {
    "model__max_depth": [None, 20],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [0.7, 1.0]
}




## Tuna den globala HistGradientBoosting-modellen



In [26]:
global_hist_pipeline = Pipeline([
    ("preprocessor", clone(global_preprocessor)),
    (
        "model",
        HistGradientBoostingRegressor(
            max_iter=300,
            l2_regularization=1.0,
            random_state=42
        )
    )
])

global_hist_search = GridSearchCV(
    estimator=global_hist_pipeline,
    param_grid=hist_parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

global_hist_search.fit(
    train_df[global_features],
    train_df["asking_price_sek"]
)

global_tuned_predictions = global_hist_search.predict(
    validation_df[global_features]
)

global_tuned_metrics = calculate_metrics(
    validation_df["asking_price_sek"],
    global_tuned_predictions
)

print("Bästa globala parametrar:", global_hist_search.best_params_)
print("Validation RMSE:", round(global_tuned_metrics["RMSE"]))




Bästa globala parametrar: {'model__learning_rate': 0.05, 'model__max_leaf_nodes': 31, 'model__min_samples_leaf': 10}
Validation RMSE: 948951


## Tuna HistGradientBoosting för lägenheter



In [27]:
apartment_hist_pipeline = Pipeline([
    ("preprocessor", clone(apartment_preprocessor)),
    (
        "model",
        HistGradientBoostingRegressor(
            max_iter=300,
            l2_regularization=1.0,
            random_state=42
        )
    )
])

apartment_hist_search = GridSearchCV(
    estimator=apartment_hist_pipeline,
    param_grid=hist_parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

apartment_hist_search.fit(
    apartment_train[apartment_features],
    apartment_train["asking_price_sek"]
)

apartment_tuned_predictions = apartment_hist_search.predict(
    apartment_validation[apartment_features]
)

apartment_tuned_metrics = calculate_metrics(
    apartment_validation["asking_price_sek"],
    apartment_tuned_predictions
)

print("Bästa lägenhetsparametrar:", apartment_hist_search.best_params_)
print("Validation RMSE:", round(apartment_tuned_metrics["RMSE"]))




Bästa lägenhetsparametrar: {'model__learning_rate': 0.05, 'model__max_leaf_nodes': 31, 'model__min_samples_leaf': 10}
Validation RMSE: 731355


## Tuna Random Forest för villor



In [28]:
house_rf_pipeline = Pipeline([
    ("preprocessor", clone(property_preprocessor)),
    (
        "model",
        RandomForestRegressor(
            n_estimators=400,
            random_state=42,
            n_jobs=-1
        )
    )
])

house_rf_search = GridSearchCV(
    estimator=house_rf_pipeline,
    param_grid=random_forest_parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

house_rf_search.fit(
    house_train[property_features],
    house_train["asking_price_sek"]
)

house_tuned_predictions = house_rf_search.predict(
    house_validation[property_features]
)

house_tuned_metrics = calculate_metrics(
    house_validation["asking_price_sek"],
    house_tuned_predictions
)

print("Bästa villaparametrar:", house_rf_search.best_params_)
print("Validation RMSE:", round(house_tuned_metrics["RMSE"]))




Bästa villaparametrar: {'model__max_depth': None, 'model__max_features': 1.0, 'model__min_samples_leaf': 1}
Validation RMSE: 1489218


## Tuna Random Forest för radhus



In [29]:
row_house_rf_pipeline = Pipeline([
    ("preprocessor", clone(property_preprocessor)),
    (
        "model",
        RandomForestRegressor(
            n_estimators=400,
            random_state=42,
            n_jobs=-1
        )
    )
])

row_house_rf_search = GridSearchCV(
    estimator=row_house_rf_pipeline,
    param_grid=random_forest_parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

row_house_rf_search.fit(
    row_house_train[property_features],
    row_house_train["asking_price_sek"]
)

row_house_tuned_predictions = row_house_rf_search.predict(
    row_house_validation[property_features]
)

row_house_tuned_metrics = calculate_metrics(
    row_house_validation["asking_price_sek"],
    row_house_tuned_predictions
)

print("Bästa radhusparametrar:", row_house_rf_search.best_params_)
print("Validation RMSE:", round(row_house_tuned_metrics["RMSE"]))




Bästa radhusparametrar: {'model__max_depth': 20, 'model__max_features': 0.7, 'model__min_samples_leaf': 1}
Validation RMSE: 979225


## Sammanställ resultatet efter tuning



In [30]:
tuning_results = pd.DataFrame([
    {
        "Segment": "GLOBAL",
        "Model": "HistGradientBoosting",
        **global_tuned_metrics
    },
    {
        "Segment": "APARTMENT",
        "Model": "HistGradientBoosting",
        **apartment_tuned_metrics
    },
    {
        "Segment": "HOUSE",
        "Model": "Random Forest",
        **house_tuned_metrics
    },
    {
        "Segment": "ROW_HOUSE",
        "Model": "Random Forest",
        **row_house_tuned_metrics
    }
])

tuning_results.round(2)

print("\nResultat efter tuning:")
print(tuning_results.round(2).to_string(index=False))





Resultat efter tuning:
  Segment                Model        MAE       RMSE  Median error   R2
   GLOBAL HistGradientBoosting  626692.18  948950.73     399898.51 0.72
APARTMENT HistGradientBoosting  460141.99  731355.22     286501.15 0.79
    HOUSE        Random Forest 1120274.99 1489217.57     885593.75 0.51
ROW_HOUSE        Random Forest  697319.86  979225.15     488025.59 0.64


## Jämför före och efter tuning

En tunad modell används bara om den faktiskt förbättrar validation-RMSE.



In [31]:
before_tuning = best_models[
    ["Segment", "RMSE"]
].rename(columns={"RMSE": "RMSE before tuning"})

after_tuning = tuning_results[
    ["Segment", "RMSE"]
].rename(columns={"RMSE": "RMSE after tuning"})

tuning_comparison = before_tuning.merge(
    after_tuning,
    on="Segment"
)

tuning_comparison["Improvement"] = (
    tuning_comparison["RMSE before tuning"]
    - tuning_comparison["RMSE after tuning"]
)

tuning_comparison.round(0)

print("\nFörändring efter tuning:")
print(tuning_comparison.round(0).to_string(index=False))





Förändring efter tuning:
  Segment  RMSE before tuning  RMSE after tuning  Improvement
APARTMENT            735208.0           731355.0       3853.0
   GLOBAL            959911.0           948951.0      10960.0
    HOUSE           1485617.0          1489218.0      -3600.0
ROW_HOUSE            978639.0           979225.0       -586.0


# Slutlig träning och test

Modellvalet är nu färdigt. Vi slår ihop train och validation, tränar om de
valda modellerna och använder sedan testdatan en enda gång.



In [32]:
final_train_df = pd.concat(
    [train_df, validation_df],
    ignore_index=True
)

final_apartment_train = final_train_df[
    final_train_df["typology"] == "APARTMENT"
].copy()

final_house_train = final_train_df[
    final_train_df["typology"] == "HOUSE"
].copy()

final_row_house_train = final_train_df[
    final_train_df["typology"] == "ROW_HOUSE"
].copy()




## Träna den slutliga globalmodellen

Vi använder den bästa tunade HistGradientBoosting-pipelinen.



In [33]:
final_global_model = clone(global_hist_search.best_estimator_)

final_global_model.fit(
    final_train_df[global_features],
    final_train_df["asking_price_sek"]
)




,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['land_area_sqm','living_area_sqm','number_rooms',...,'has_land_area', 'municipality','typology']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically pa

## Träna den slutliga lägenhetsmodellen



In [34]:
final_apartment_model = clone(apartment_hist_search.best_estimator_)

final_apartment_model.fit(
    final_apartment_train[apartment_features],
    final_apartment_train["asking_price_sek"]
)




,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['living_area_sqm','number_rooms','latitude','longitude','municipality']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset

## Träna den slutliga villamodellen

Tuningen förbättrade inte validation-RMSE. Därför behåller vi den enklare
Random Forest-konfigurationen från screeningen.



In [35]:
final_house_model = Pipeline([
    ("preprocessor", clone(property_preprocessor)),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])

final_house_model.fit(
    final_house_train[property_features],
    final_house_train["asking_price_sek"]
)




,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['land_area_sqm','living_area_sqm','number_rooms',...,'longitude', 'has_land_area','municipality']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically p

## Träna den slutliga radhusmodellen



In [36]:
final_row_house_model = Pipeline([
    ("preprocessor", clone(property_preprocessor)),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])

final_row_house_model.fit(
    final_row_house_train[property_features],
    final_row_house_train["asking_price_sek"]
)




,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['land_area_sqm','living_area_sqm','number_rooms',...,'longitude', 'has_land_area','municipality']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically p

## Utvärdera slutmodellerna på testdata



In [37]:
apartment_test = test_df[test_df["typology"] == "APARTMENT"].copy()
house_test = test_df[test_df["typology"] == "HOUSE"].copy()
row_house_test = test_df[test_df["typology"] == "ROW_HOUSE"].copy()

global_test_predictions = final_global_model.predict(
    test_df[global_features]
)

apartment_test_predictions = final_apartment_model.predict(
    apartment_test[apartment_features]
)

house_test_predictions = final_house_model.predict(
    house_test[property_features]
)

row_house_test_predictions = final_row_house_model.predict(
    row_house_test[property_features]
)

global_test_metrics = calculate_metrics(
    test_df["asking_price_sek"],
    global_test_predictions
)

apartment_test_metrics = calculate_metrics(
    apartment_test["asking_price_sek"],
    apartment_test_predictions
)

house_test_metrics = calculate_metrics(
    house_test["asking_price_sek"],
    house_test_predictions
)

row_house_test_metrics = calculate_metrics(
    row_house_test["asking_price_sek"],
    row_house_test_predictions
)

final_test_results = pd.DataFrame([
    {
        "Segment": "GLOBAL",
        "Model": "HistGradientBoosting",
        **global_test_metrics
    },
    {
        "Segment": "APARTMENT",
        "Model": "HistGradientBoosting",
        **apartment_test_metrics
    },
    {
        "Segment": "HOUSE",
        "Model": "Random Forest",
        **house_test_metrics
    },
    {
        "Segment": "ROW_HOUSE",
        "Model": "Random Forest",
        **row_house_test_metrics
    }
])

final_test_results.round(2)

print("\nSlutresultat på testdata:")
print(final_test_results.round(2).to_string(index=False))





Slutresultat på testdata:
  Segment                Model        MAE       RMSE  Median error   R2
   GLOBAL HistGradientBoosting  620692.90  914743.86     401603.51 0.76
APARTMENT HistGradientBoosting  459795.15  700756.76     294863.76 0.80
    HOUSE        Random Forest 1093516.24 1434707.16     880150.00 0.60
ROW_HOUSE        Random Forest  830374.24 1206367.39     551433.33 0.51


## Jämför global och specialiserad modell på samma testbostäder

Den globala modellen utvärderas separat på lägenheter, villor och radhus.
Då jämförs modellerna på exakt samma testobjekt.



In [38]:
global_apartment_predictions = final_global_model.predict(
    apartment_test[global_features]
)

global_house_predictions = final_global_model.predict(
    house_test[global_features]
)

global_row_house_predictions = final_global_model.predict(
    row_house_test[global_features]
)

global_vs_specialized = pd.DataFrame([
    {
        "Segment": "APARTMENT",
        "Model": "Global",
        **calculate_metrics(
            apartment_test["asking_price_sek"],
            global_apartment_predictions
        )
    },
    {
        "Segment": "APARTMENT",
        "Model": "Specialized",
        **calculate_metrics(
            apartment_test["asking_price_sek"],
            apartment_test_predictions
        )
    },
    {
        "Segment": "HOUSE",
        "Model": "Global",
        **calculate_metrics(
            house_test["asking_price_sek"],
            global_house_predictions
        )
    },
    {
        "Segment": "HOUSE",
        "Model": "Specialized",
        **calculate_metrics(
            house_test["asking_price_sek"],
            house_test_predictions
        )
    },
    {
        "Segment": "ROW_HOUSE",
        "Model": "Global",
        **calculate_metrics(
            row_house_test["asking_price_sek"],
            global_row_house_predictions
        )
    },
    {
        "Segment": "ROW_HOUSE",
        "Model": "Specialized",
        **calculate_metrics(
            row_house_test["asking_price_sek"],
            row_house_test_predictions
        )
    }
])

global_vs_specialized.round(2)

print("\nGlobal mot specialiserad modell på samma testdata:")
print(global_vs_specialized.round(2).to_string(index=False))





Global mot specialiserad modell på samma testdata:
  Segment       Model        MAE       RMSE  Median error   R2
APARTMENT      Global  476846.21  709211.89     320641.80 0.79
APARTMENT Specialized  459795.15  700756.76     294863.76 0.80
    HOUSE      Global 1045104.46 1364085.35     804230.89 0.64
    HOUSE Specialized 1093516.24 1434707.16     880150.00 0.60
ROW_HOUSE      Global  713185.78  945301.75     599108.21 0.70
ROW_HOUSE Specialized  830374.24 1206367.39     551433.33 0.51


# Spara modellerna

Varje fil innehåller tre delar:

- `pipeline`: preprocessing och den tränade modellen
- `metrics`: modellens slutresultat på testdatan
- `metadata`: information om hur modellen skapades



In [39]:
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)


def prepare_metrics_for_saving(metrics):
    return {
        "mae": float(metrics["MAE"]),
        "rmse": float(metrics["RMSE"]),
        "median_error": float(metrics["Median error"]),
        "r2": float(metrics["R2"])
    }




## Skapa bundle för globalmodellen



In [40]:
global_bundle = {
    "pipeline": final_global_model,
    "metrics": prepare_metrics_for_saving(global_test_metrics),
    "metadata": {
        "segment": "GLOBAL",
        "model_name": "HistGradientBoostingRegressor",
        "training_rows": len(final_train_df),
        "test_rows": len(test_df),
        "feature_columns": global_features,
        "target": "asking_price_sek",
        "price_min": 100_000,
        "price_max": 10_000_000,
        "random_state": 42,
        "data_file": DATA_PATH.name,
        "sklearn_version": sklearn.__version__
    }
}




## Skapa bundle för lägenhetsmodellen



In [41]:
apartment_bundle = {
    "pipeline": final_apartment_model,
    "metrics": prepare_metrics_for_saving(apartment_test_metrics),
    "metadata": {
        "segment": "APARTMENT",
        "model_name": "HistGradientBoostingRegressor",
        "training_rows": len(final_apartment_train),
        "test_rows": len(apartment_test),
        "feature_columns": apartment_features,
        "target": "asking_price_sek",
        "price_min": 100_000,
        "price_max": 10_000_000,
        "random_state": 42,
        "data_file": DATA_PATH.name,
        "sklearn_version": sklearn.__version__
    }
}




## Skapa bundle för villamodellen



In [42]:
house_bundle = {
    "pipeline": final_house_model,
    "metrics": prepare_metrics_for_saving(house_test_metrics),
    "metadata": {
        "segment": "HOUSE",
        "model_name": "RandomForestRegressor",
        "training_rows": len(final_house_train),
        "test_rows": len(house_test),
        "feature_columns": property_features,
        "target": "asking_price_sek",
        "price_min": 100_000,
        "price_max": 10_000_000,
        "random_state": 42,
        "data_file": DATA_PATH.name,
        "sklearn_version": sklearn.__version__
    }
}




## Skapa bundle för radhusmodellen



In [43]:
row_house_bundle = {
    "pipeline": final_row_house_model,
    "metrics": prepare_metrics_for_saving(row_house_test_metrics),
    "metadata": {
        "segment": "ROW_HOUSE",
        "model_name": "RandomForestRegressor",
        "training_rows": len(final_row_house_train),
        "test_rows": len(row_house_test),
        "feature_columns": property_features,
        "target": "asking_price_sek",
        "price_min": 100_000,
        "price_max": 10_000_000,
        "random_state": 42,
        "data_file": DATA_PATH.name,
        "sklearn_version": sklearn.__version__
    }
}




## Skriv modellfilerna till models-mappen

`compress=3` minskar filstorleken utan att göra sparandet onödigt långsamt.



In [44]:
joblib.dump(
    global_bundle,
    MODELS_DIR / "global_model.joblib",
    compress=3
)

joblib.dump(
    apartment_bundle,
    MODELS_DIR / "apartment_model.joblib",
    compress=3
)

joblib.dump(
    house_bundle,
    MODELS_DIR / "house_model.joblib",
    compress=3
)

joblib.dump(
    row_house_bundle,
    MODELS_DIR / "row_house_model.joblib",
    compress=3
)

print("\nSparade modellfiler:")
for model_path in sorted(MODELS_DIR.glob("*_model.joblib")):
    size_mb = model_path.stat().st_size / 1_000_000
    print(f"{model_path.name}: {size_mb:.1f} MB")



Sparade modellfiler:
apartment_model.joblib: 0.4 MB
global_model.joblib: 0.4 MB
house_model.joblib: 10.4 MB
row_house_model.joblib: 2.6 MB
